# 01 Scope and data inventory

Builds a plain inventory of every dataset the later steps need: what is confirmed to exist, what licence it carries, whether it has been downloaded, and whether it has actually been read yet. This notebook does not analyze the resort or the city, it only reports on the state of the evidence behind steps 02-08.

In [ ]:
research_dir = "research"
output_dir = "data/processed"

## Load the ledger

research/sources.csv and research/claims.csv are the only inputs. src/resort/ledger.py's read_ledger() cross-checks row counts against ID-prefixed line counts before returning anything, so a CSV quoting error would raise here rather than silently produce a short inventory.

In [ ]:
import sys

sys.path.insert(0, "src")
from resort.ledger import read_ledger

ledger = read_ledger(
    claims_path=f"{research_dir}/claims.csv",
    sources_path=f"{research_dir}/sources.csv",
)
sources = ledger["sources"]
claims = ledger["claims"]
print(f"{len(sources)} sources, {len(claims)} claims")

## Dataset inventory table

One row per source: id, type, title, licence, whether a local copy has been downloaded (data/raw), and verified_exists as last set by tools/verify_sources.py. This is the table steps/01 points to when it says what evidence each later step currently has.

In [ ]:
import pandas as pd

inventory = pd.DataFrame(
    [
        {
            "id": s["id"],
            "type": s["type"],
            "title": s["title"],
            "licence": s["licence"] or "(not recorded)",
            "has_local_copy": bool(s["local_copy"].strip()),
            "verified_exists": s["verified_exists"] or "(unchecked)",
        }
        for s in sources.values()
    ]
).sort_values("id")
inventory

## Claims by step and status

Every claim is still needs-review or (once source-verifier runs) agent-checked; none are human-verified yet, so nothing here can be cited in the report (step 09) until that changes. This table is a plain count, not a judgment about which claims are trustworthy.

In [ ]:
claims_by_step = pd.DataFrame(
    [{"step": c["step"], "status": c["status"]} for c in claims.values()]
)
status_counts = (
    claims_by_step.groupby(["step", "status"]).size().unstack(fill_value=0).sort_index()
)
status_counts

## Write outputs

The dataset inventory and status counts go to data/processed so other notebooks and the eventual report can read them without re-deriving them, and so no number here is ever typed by hand into a step file or the report.

In [ ]:
import json
import os

os.makedirs(output_dir, exist_ok=True)
inventory.to_csv(f"{output_dir}/01_dataset_inventory.csv", index=False, encoding="utf-8")
status_counts.to_csv(f"{output_dir}/01_claims_by_step_status.csv", encoding="utf-8")

summary = {
    "n_sources": len(sources),
    "n_claims": len(claims),
    "n_sources_with_local_copy": int(inventory["has_local_copy"].sum()),
    "n_claims_human_verified": int((claims_by_step["status"] == "human-verified").sum()),
}
with open(f"{output_dir}/01_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
summary

## Checks

A passing run must show at least one source and one claim (the ledger guard already raises if the CSVs are unreadable), and the human-verified count must be an int the report notebook can use directly rather than a number someone typed by hand.

In [ ]:
assert summary["n_sources"] > 0, "no sources loaded"
assert summary["n_claims"] > 0, "no claims loaded"
assert isinstance(summary["n_claims_human_verified"], int)
print("checks passed")
print(f"human-verified claims so far: {summary['n_claims_human_verified']} of {summary['n_claims']}")

## Versions

In [ ]:
import importlib.metadata
import sys

print("python", sys.version)
for pkg in ["pandas"]:
    print(pkg, importlib.metadata.version(pkg))